# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Model Choice

I selected **Random Forest Classifier** because it can learn non-linear relationships between features, reduces overfitting by combining multiple decision trees, and provides feature importance for interpretation. It is suitable for comparing with my Week 4 rule-based baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Recreate baseline score
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 1000
ctr_gap = (
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & (df["ctr"] < 2)
)

df["baseline_score"] = (
    2 * stale.astype(int)
    + 2 * visible.astype(int)
    + 3 * ctr_gap.astype(int)
)

# Target
df["target"] = (df["baseline_score"] >= 5).astype(int)

# Features
X = df[
    [
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
    ]
]

y = df["target"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(X_train.shape)
print(X_test.shape)

(24000, 4)
(6000, 4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# ----- Week 4 Rule Baseline -----
baseline_pred = (
    (
        2 * (X_test["days_since_last_update"] >= 180).astype(int)
        + 2 * (X_test["impressions_90d"] >= 1000).astype(int)
        + 3 * (
            (X_test["avg_position"] > 0)
            & (X_test["avg_position"] <= 10)
            & (X_test["ctr"] < 2)
        ).astype(int)
    ) >= 5
).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

# ----- Random Forest -----
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

rf_pred = model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

comparison = pd.DataFrame({
    "Model": ["Week 4 Rule Baseline", "Random Forest"],
    "Accuracy": [baseline_accuracy, rf_accuracy]
})

comparison

,Model,Accuracy
0,Week 4 Rule Baseline,1.0
1,Random Forest,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Random Forest Accuracy:", rf_accuracy)

print("\nClassification Report")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, rf_pred))

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values("Importance", ascending=False)

Random Forest Accuracy: 1.0

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4675
           1       1.00      1.00      1.00      1325

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000


Confusion Matrix
[[4675    0]
 [   0 1325]]


,Feature,Importance
2,avg_position,0.524698
1,impressions_90d,0.391910
3,ctr,0.073349
0,days_since_last_update,0.010043


### Interpretation

The Random Forest model was evaluated using the same data split and metric as the Week 4 rule-based baseline. The comparison table shows whether the machine learning model improves over the handcrafted rules. Feature importance highlights which signals contribute most to the model's decisions, while the confusion matrix and classification report help identify where prediction errors occur.

## Self-check

- I selected Random Forest because it is robust and provides feature importance.
- I used the same dataset as Week 4.
- I evaluated both the rule-based baseline and the Random Forest model on the same test split.
- I compared both models using accuracy.
- I reviewed feature importance and analyzed prediction errors.